In [0]:
import pytest
from datetime import datetime
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F

def test_trip_duration_calculation(spark):
    df = spark.createDataFrame([
        Row(trip_id="T1",
            pickup_datetime=datetime(2026, 1, 1, 10, 0, 0),
            dropoff_datetime=datetime(2026, 1, 1, 10, 15, 0))
    ])
    result = df.withColumn(
        "trip_duration_minutes",
        F.round((F.col("dropoff_datetime").cast("long") - F.col("pickup_datetime").cast("long")) / 60.0, 2)
    ).collect()
    assert result[0]["trip_duration_minutes"] == 15.0


def test_revenue_per_km_calculation(spark):
    df = spark.createDataFrame([
        Row(trip_id="T1", distance_km=10.0, total_amount=25.0)
    ])
    result = df.withColumn(
        "revenue_per_km",
        F.round(F.col("total_amount") / F.col("distance_km"), 2)
    ).collect()
    assert result[0]["revenue_per_km"] == 2.5

In [0]:
def run_test(test_func, name):
    try:
        test_func(spark)
        print(f"✅ PASS: {name}")
    except AssertionError as e:
        print(f"❌ FAIL: {name} — {e}")
    except Exception as e:
        print(f"⚠️ ERROR: {name} — {e}")

run_test(test_trip_duration_calculation, "test_trip_duration_calculation")
run_test(test_revenue_per_km_calculation, "test_revenue_per_km_calculation")